<a href="https://colab.research.google.com/github/aqwenaaa/Model-Comparison/blob/valina-branch/23_Nervalina_Adzra_JS4/7_SBP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. LOAD DATA**

Data klaim berisi informasi transaksi klaim, sedangkan data polis berisi data nasabah.

In [ ]:
import pandas as pd

# Load dataset dari GitHub
url_klaim = "https://raw.githubusercontent.com/valinadz/Dataset-Asuransi/refs/heads/main/Data_Klaim.csv"
url_polis = "https://raw.githubusercontent.com/valinadz/Dataset-Asuransi/refs/heads/main/Data_Polis.csv"

df_klaim = pd.read_csv(url_klaim)
df_polis = pd.read_csv(url_polis)

print("Data Klaim:")
display(df_klaim.head())

print("Data Polis:")
display(df_polis.head())

Data Klaim:


,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS
0,C-0001-M,POL-0176,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,2024-07-08,2024-05-27,2024-05-27,28093653.0,6.143948e+06,Singapore
1,C-0002-M,POL-3288,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-08-06,2024-07-15,2024-07-15,80987278.0,8.230952e+07,Malaysia
2,C-0003-M,POL-1786,R,OP,C18.9,"MALIGNANT NEOPLASM, COLON, UNSPECIFIED",PAID,2024-10-17,2024-05-16,2024-05-16,183047130.0,1.928599e+08,Singapore
3,C-0004-M,POL-1786,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-09-03,2024-07-18,2024-07-18,191424386.0,1.914244e+08,Singapore
4,C-0005-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaN,2024-06-06,2024-06-06,138936357.0,1.389364e+08,Singapore


Data Polis:


,Nomor Polis,Plan Code,Gender,Tanggal Lahir,Tanggal Efektif Polis,Domisili
0,POL-0001,M-003,M,19640811,20140603,JAKARTA
1,POL-0002,M-003,M,19710730,20140603,JAKARTA
2,POL-0003,M-001,M,19790821,20160808,JAKARTA
3,POL-0004,M-003,M,20140724,20160811,JAKARTA
4,POL-0005,M-001,F,19810114,20150828,JAKARTA


**2. PREPROCESSING (AMBIL DATA)**

In [ ]:
# Ambil 1 data sebagai contoh analisis
data = df_klaim.iloc[0]

print(data)

Claim ID                                             C-0001-M
Nomor Polis                                          POL-0176
Reimburse/Cashless                                          R
Inpatient/Outpatient                                       OP
ICD Diagnosis                                             C50
ICD Description                  MALIGNANT NEOPLASM OF BREAST
Status Klaim                                             PAID
Tanggal Pembayaran Klaim                           2024-07-08
Tanggal Pasien Masuk RS                            2024-05-27
Tanggal Pasien Keluar RS                           2024-05-27
Nominal Klaim Yang Disetujui                       28093653.0
Nominal Biaya RS Yang Terjadi                      6143947.68
Lokasi RS                                           Singapore
Name: 0, dtype: object


**3. PEMBENTUKAN FAKTA (KNOWLEDGE BASE)**

In [ ]:
facts = {
    "klaim_valid": data["Status Klaim"] == "PAID",
    "klaim_besar": data["Nominal Klaim Yang Disetujui"] > 50000000,
    "luar_negeri": data["Lokasi RS"] != "Indonesia"
}

print("Fakta:", facts)

Fakta: {'klaim_valid': True, 'klaim_besar': np.False_, 'luar_negeri': True}


Fakta adalah kondisi awal yang digunakan dalam sistem pakar:

klaim_valid → klaim sudah dibayar

klaim_besar → nominal tinggi

luar_negeri → lokasi RS luar Indonesia

**4. FORWARD CHAINING**

In [ ]:
rules = [
    (["klaim_valid", "klaim_besar"], "klaim_disetujui"),
    (["luar_negeri", "klaim_besar"], "risiko_tinggi")
]

def forward_chaining(facts, rules):
    inferred = facts.copy()
    changed = True

    while changed:
        changed = False
        for kondisi, hasil in rules:
            if all(inferred.get(k, False) for k in kondisi):
                if hasil not in inferred:
                    inferred[hasil] = True
                    changed = True
    return inferred

hasil_fc = forward_chaining(facts, rules)
print("Hasil Forward Chaining:", hasil_fc)

Hasil Forward Chaining: {'klaim_valid': True, 'klaim_besar': np.False_, 'luar_negeri': True}


Forward chaining adalah metode data-driven:

*   Dimulai dari fakta
*   Mencocokkan rule
*   Menghasilkan kesimpulan baru

➡️ Contoh:

Jika klaim valid + nominal besar → klaim disetujui

**5. BACKWARD CHAINING**

In [ ]:
def backward_chaining(goal, facts, rules):
    if goal in facts:
        return facts[goal]

    for kondisi, hasil in rules:
        if hasil == goal:
            if all(backward_chaining(k, facts, rules) for k in kondisi):
                return True
    return False

goal = "klaim_disetujui"
hasil_bc = backward_chaining(goal, facts, rules)

print("Hasil Backward Chaining:", hasil_bc)

Hasil Backward Chaining: False


Backward chaining adalah metode goal-driven:

*   Dimulai dari tujuan (goal)
*   Mengecek apakah syarat terpenuhi

➡️ Cocok untuk:

*   Validasi klaim
*   Diagnosis


**6. TEOREMA BAYES**

In [ ]:
# Probabilitas klaim disetujui
P_klaim = len(df_klaim[df_klaim["Status Klaim"] == "PAID"]) / len(df_klaim)

# Probabilitas klaim besar
P_besar = len(df_klaim[df_klaim["Nominal Klaim Yang Disetujui"] > 50000000]) / len(df_klaim)

# Likelihood
P_besar_given_klaim = len(df_klaim[
    (df_klaim["Status Klaim"] == "PAID") &
    (df_klaim["Nominal Klaim Yang Disetujui"] > 50000000)
]) / len(df_klaim[df_klaim["Status Klaim"] == "PAID"])

# Bayes
P_klaim_given_besar = (P_besar_given_klaim * P_klaim) / P_besar

print("Probabilitas klaim disetujui jika nominal besar:", P_klaim_given_besar)

Probabilitas klaim disetujui jika nominal besar: 1.0


Teorema Bayes digunakan untuk menghitung probabilitas:

P(Klaim | Besar) =

(P(Besar | Klaim) × P(Klaim)) / P(Besar)

➡️ Digunakan untuk:

*   Analisis risiko klaim
*   Prediksi keputusan

**7. CERTAINTY FACTOR**

In [ ]:
# Nilai dari pakar
cf_pakar = {
    "klaim_valid": 0.9,
    "klaim_besar": 0.8,
    "luar_negeri": 0.6
}

# Nilai dari data (user)
cf_user = {
    "klaim_valid": 1 if facts["klaim_valid"] else 0,
    "klaim_besar": 1 if facts["klaim_besar"] else 0,
    "luar_negeri": 1 if facts["luar_negeri"] else 0
}

# Hitung CF tiap evidence
cf_evidence = {}
for key in cf_pakar:
    cf_evidence[key] = cf_pakar[key] * cf_user[key]

# Fungsi kombinasi CF
def combine_cf(cf1, cf2):
    return cf1 + cf2 * (1 - cf1)

# Gabungkan semua CF
cf_total = 0
for val in cf_evidence.values():
    cf_total = combine_cf(cf_total, val)

print("Nilai Certainty Factor:", cf_total)

Nilai Certainty Factor: 0.96


Certainty Factor digunakan untuk mengukur tingkat keyakinan:

CF = MB - MD

➡️ Dalam implementasi:

*   CF pakar = tingkat kepercayaan ahli
*   CF user = kondisi dari data
*   Hasil = tingkat keyakinan sistem terhadap keputusan


Berdasarkan hasil implementasi:

1. Forward chaining digunakan untuk menghasilkan keputusan dari fakta yang ada.

2. Backward chaining digunakan untuk memverifikasi apakah suatu keputusan dapat dicapai.

3. Teorema Bayes digunakan untuk menghitung probabilitas keputusan berdasarkan data historis.

4. Certainty factor digunakan untuk menentukan tingkat keyakinan terhadap keputusan tersebut.